# Compositionality Shared Task - Fine-tune ModernBERT (5-Fold)

> **LEGACY** - uses the pre-marker 4-sequence architecture and is now superseded by `main.py` / `notebook/run_main.ipynb`.

**Modules:** shared in `src/` (dataset, model, loss, train, folds) - single source of truth for both notebooks.

In [ ]:
import os
import sys
import glob
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from sklearn.metrics import mean_squared_error
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.amp import autocast, GradScaler
from transformers import AutoTokenizer, get_linear_schedule_with_warmup


def _setup_src():
    # Local runs: notebook sits next to src/ (repo root) or one level up
    for base in ('.', '..'):
        if os.path.isdir(os.path.join(base, 'src')):
            sys.path.insert(0, os.path.abspath(base))
            return
    # Kaggle runs: GitHub repo mounted under /kaggle/input/<repo>/src
    hits = sorted(glob.glob('/kaggle/input/*/src') + glob.glob('/kaggle/input/**/src', recursive=True))
    if hits:
        sys.path.insert(0, os.path.dirname(hits[0]))
        return
    raise RuntimeError('src/ not found. Add the GitHub repo as a Kaggle input (Add Input -> GitHub).')


_setup_src()

from src.constants import MODEL_NAME, MAX_LENGTH, MAX_CONTEXT_LENGTH
from src.dataset import NNDataset
from src.model import ModernBERTRegressor
from src.loss import CombinedLoss
from src.train import train_epoch, evaluate, unfreeze_top_layers

KAGGLE_PATH = '/kaggle/input/datasets/ieltsmater/compartment/Compartment'
OUTPUT_DIR = '/kaggle/working'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 1. Load Data & Create Folds

In [ ]:
from src.folds import prepare_stratified_folds

In [ ]:
df = pd.read_csv(f'{KAGGLE_PATH}/dataset/en-nn-train.tsv', sep='\t')
df = prepare_stratified_folds(df, target_col='Compound')

print(f'Dataset: {df.shape}')
print(f'Unique compounds: {df["Compound"].nunique()}')
print(f'\nRows per fold:')
print(df.groupby('fold').size())

## 2. Dataset & Model

In [ ]:
# Dataset & model definitions live in src/ (single source of truth)
from src.constants import MODEL_NAME, MAX_LENGTH, MAX_CONTEXT_LENGTH
from src.dataset import NNDataset
from src.model import ModernBERTRegressor
from src.loss import CombinedLoss

# Training helpers live in src/train.py
from src.train import train_epoch, evaluate, unfreeze_top_layers

## 3. Shared Training Functions

`train_epoch`, `evaluate`, `unfreeze_top_layers` live in `src/train.py` and are imported in the setup cell.

## 4. 5-Fold Training

In [ ]:
# Hyperparameters - Final Optimized Config
CONFIG = {
    'model_name': MODEL_NAME,
    'max_length': MAX_LENGTH,
    'max_context_length': MAX_CONTEXT_LENGTH,
    'batch_size': 32,
    'head_lr': 5e-4,            # Giảm nhẹ head_lr để khớp với Batch Size 16
    'encoder_lr': 3e-6,         # SỬA: Giảm sâu để bảo vệ ModernBERT khi unfreeze 10 layers
    'weight_decay': 0.05,       # SỬA: Tăng chút chống overfitting
    'num_epochs': 10,
    'freeze_epochs': 5,         # Freeze 2 epochs đầu cho Head học mượt
    'unfreeze_from_layer': 19,  # Unfreeze layers 12-21 (10 layers)
    'warmup_ratio': 0.15,       # SỬA: Warmup mượt hơn cho giai đoạn chuyển pha
    'dropout': 0.2,             # SỬA: Tăng chống ghi nhớ vẹt
    'loss_type': 'mse_ccc',
    'ccc_weight': 0.7,          # Tối ưu trực tiếp chỉ số Ranking (CCC/Spearman)
    'patience': 5,              # Early stopping mỗi fold, chỉ đếm từ phase unfreeze
}

print('Config:', CONFIG)

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.amp import autocast, GradScaler
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from scipy.stats import spearmanr

# Tokenizer & Storage
tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])
oof_mod = np.zeros(len(df))
oof_head = np.zeros(len(df))
completed_folds = []
fold_results = []

# Đảm bảo dùng CombinedLoss đã định nghĩa
criterion = CombinedLoss(ccc_weight=CONFIG.get('ccc_weight', 0.7))

for fold in range(5):
    print(f'\n{"="*50}\nFOLD {fold}\n{"="*50}')
    
    train_df = df[df['fold'] != fold].reset_index(drop=True)
    val_df = df[df['fold'] == fold].reset_index(drop=True)
    
    train_dataset = NNDataset(train_df, tokenizer, max_context_length=CONFIG['max_context_length'])
    val_dataset = NNDataset(val_df, tokenizer, max_context_length=CONFIG['max_context_length'])
    
    train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size']*2, shuffle=False, num_workers=2, pin_memory=True)
    
    model = ModernBERTRegressor(CONFIG['model_name'], dropout=CONFIG['dropout'], freeze_bert=True).to(device)
    scaler = GradScaler('cuda')
    
    frozen_epochs = CONFIG['freeze_epochs']
    unfrozen_epochs = CONFIG['num_epochs'] - frozen_epochs
    
    # 1. FIX: Thu thập thông số của CẢ 2 NHÁNH Regressor (Phase 1)
    head_params = list(model.mod_regressor.parameters()) + list(model.head_regressor.parameters())
    optimizer = torch.optim.AdamW(head_params, lr=CONFIG['head_lr'], weight_decay=CONFIG['weight_decay'])
    
    scheduler = get_linear_schedule_with_warmup(
        optimizer, 
        num_warmup_steps=int(len(train_loader) * frozen_epochs * CONFIG['warmup_ratio']),
        num_training_steps=len(train_loader) * frozen_epochs
    )
    
    best_rho = -1.0
    best_epoch = -1
    best_mod_pred, best_head_pred = None, None
    best_mod_label, best_head_label = None, None
    no_improve_epochs = 0
    
    for epoch in range(CONFIG['num_epochs']):
        # --- Unfreeze Phase 2 ---
        if epoch == frozen_epochs:
            print(f'\n  >>> Unfreezing top layers at epoch {epoch+1} <<<')
            unfreeze_top_layers(model, CONFIG['unfreeze_from_layer'])
            
            # 2. FIX: Tách tham số Encoder và 2 nhánh Regressor (Phase 2)
            encoder_params = [p for n, p in model.named_parameters() 
                              if 'mod_regressor' not in n and 'head_regressor' not in n and p.requires_grad]
            head_params = list(model.mod_regressor.parameters()) + list(model.head_regressor.parameters())
            
            optimizer = torch.optim.AdamW([
                {'params': encoder_params, 'lr': CONFIG['encoder_lr']},
                {'params': head_params, 'lr': CONFIG['head_lr']}
            ], weight_decay=CONFIG['weight_decay'])
            
            scheduler = get_linear_schedule_with_warmup(
                optimizer, 
                num_warmup_steps=int(len(train_loader) * unfrozen_epochs * CONFIG['warmup_ratio']),
                num_training_steps=len(train_loader) * unfrozen_epochs
            )
            
        phase = 'FROZEN' if epoch < frozen_epochs else 'UNFROZEN-TOP'
        
        # --- Training ---
        train_loss = train_epoch(model, train_loader, optimizer, scheduler, criterion, scaler, device)
        
        # --- Evaluate ---
        mod_pred, head_pred, mod_label, head_label = evaluate(model, val_loader, device)
        
        rho_mod = spearmanr(mod_label, mod_pred).statistic
        rho_head = spearmanr(head_label, head_pred).statistic
        rho_mean = (rho_mod + rho_head) / 2
        
        print(f'  Epoch {epoch+1}/{CONFIG["num_epochs"]} [{phase}] | '
              f'Loss: {train_loss:.4f} | '
              f'Mod ρ: {rho_mod:.4f} | Head ρ: {rho_head:.4f} | Mean ρ: {rho_mean:.4f}')
        
        # Save Best Checkpoint
        if rho_mean > best_rho:
            best_rho = rho_mean
            best_epoch = epoch + 1
            best_mod_pred = mod_pred
            best_head_pred = head_pred
            best_mod_label = mod_label
            best_head_label = head_label
            no_improve_epochs = 0
            
            os.makedirs(f'{OUTPUT_DIR}/models', exist_ok=True)
            torch.save(model.state_dict(), f'{OUTPUT_DIR}/models/fold{fold}_best.pt')
        else:
            # Early stopping: chỉ đếm từ phase unfreeze (epoch >= frozen_epochs)
            if epoch >= frozen_epochs:
                no_improve_epochs += 1
                print(f'  -- No improvement {no_improve_epochs}/{CONFIG["patience"]} epochs')
                if no_improve_epochs >= CONFIG['patience']:
                    print(f'  ==> Early stop fold {fold} at epoch {epoch+1} ' +
                          f'(best = epoch {best_epoch}, Mean ρ = {best_rho:.4f})')
                    break
            
    # ----------------------------------------------------
    # LƯU OOF CHO FOLD HIỆN TẠI (GIỮ NGUYÊN GIÁ TRỊ GỐC)
    # ----------------------------------------------------
    val_indices = df[df['fold'] == fold].index.values
    
    oof_mod[val_indices] = best_mod_pred
    oof_head[val_indices] = best_head_pred
    completed_folds.append(fold)
    
    fold_rho_mod = spearmanr(best_mod_label, best_mod_pred).statistic
    fold_rho_head = spearmanr(best_head_label, best_head_pred).statistic
    
    fold_results.append({
        'fold': fold, 
        'rho_mod': fold_rho_mod, 
        'rho_head': fold_rho_head, 
        'rho_mean': (fold_rho_mod + fold_rho_head) / 2
    })
    
    # ----------------------------------------------------
    # TÍNH TOÁN OOF THỰC TẾ CHO CÁC FOLD ĐÃ HOÀN THÀNH
    # ----------------------------------------------------
    active_idx = df['fold'].isin(completed_folds)
    real_oof_mod = spearmanr(df.loc[active_idx, 'ModAvg'], oof_mod[active_idx]).statistic
    real_oof_head = spearmanr(df.loc[active_idx, 'HeadAvg'], oof_head[active_idx]).statistic
    
    print(f'\n  >>> FOLD {fold} Finished | Best Mean ρ: {best_rho:.4f}')
    print(f'  >>> REAL OOF (Folds {completed_folds}) Mean ρ: {(real_oof_mod + real_oof_head)/2:.4f}')
    
    del model, optimizer, scheduler
    torch.cuda.empty_cache()

## 5. Results Summary

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from sklearn.metrics import mean_squared_error

# 1. Bảng kết quả từng Fold
results_df = pd.DataFrame(fold_results)
print('=== Per-Fold Results ===')
print(results_df.to_string(index=False))

# 2. Trung bình các Fold
print(f'\n=== Fold Averages ===')
print(f'Mod ρ  (mean ± std): {results_df["rho_mod"].mean():.4f} ± {results_df["rho_mod"].std():.4f}')
print(f'Head ρ (mean ± std): {results_df["rho_head"].mean():.4f} ± {results_df["rho_head"].std():.4f}')
print(f'Mean ρ (overall)   : {results_df["rho_mean"].mean():.4f}')

# 3. Overall Out-Of-Fold (OOF) Metrics
oof_rho_mod = spearmanr(df['ModAvg'], oof_mod).statistic
oof_rho_head = spearmanr(df['HeadAvg'], oof_head).statistic
oof_rho_mean = (oof_rho_mod + oof_rho_head) / 2

rmse_mod = np.sqrt(mean_squared_error(df['ModAvg'], oof_mod))
rmse_head = np.sqrt(mean_squared_error(df['HeadAvg'], oof_head))

print(f'\n=== Overall OOF Score ===')
print(f'OOF Mod ρ  : {oof_rho_mod:.4f}')
print(f'OOF Head ρ : {oof_rho_head:.4f}')
print(f'🔥 OOF Mean ρ: {oof_rho_mean:.4f}')

print(f'\n=== Overall OOF RMSE ===')
print(f'OOF RMSE Mod : {rmse_mod:.4f}')
print(f'OOF RMSE Head: {rmse_head:.4f}')

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Biểu đồ cho Modifier Compositionality
axes[0].hist(df['ModAvg'], bins=30, alpha=0.5, label='Gold', density=True, color='blue')
axes[0].hist(oof_mod, bins=30, alpha=0.5, label='Predicted', density=True, color='orange')
axes[0].set_title('Modifier Compositionality Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Score (0 - 5)', fontsize=10)
axes[0].set_ylabel('Density', fontsize=10)
axes[0].set_xlim(0, 5)
axes[0].grid(True, linestyle='--', alpha=0.5)
axes[0].legend()

# 2. Biểu đồ cho Head Compositionality
axes[1].hist(df['HeadAvg'], bins=30, alpha=0.5, label='Gold', density=True, color='blue')
axes[1].hist(oof_head, bins=30, alpha=0.5, label='Predicted', density=True, color='orange')
axes[1].set_title('Head Compositionality Distribution', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Score (0 - 5)', fontsize=10)
axes[1].set_ylabel('Density', fontsize=10)
axes[1].set_xlim(0, 5)
axes[1].grid(True, linestyle='--', alpha=0.5)
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Predict on Trial & Generate Submission

In [ ]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from scipy.stats import spearmanr
from sklearn.metrics import mean_squared_error

# 1. Load trial data
df_trial = pd.read_csv(f'{KAGGLE_PATH}/trial/en-nn-trial.tsv', sep='\t')

# Khởi tạo Dataset an toàn với cờ is_test=True (hoặc để tự nhận diện)
trial_dataset = NNDataset(
    df_trial, 
    tokenizer, 
    max_context_length=CONFIG['max_context_length'],
    is_test=True
)
trial_loader = DataLoader(
    trial_dataset, 
    batch_size=CONFIG['batch_size'] * 2, 
    shuffle=False, 
    num_workers=2,
    pin_memory=True
)

# 2. Ensemble predictions từ cả 5 Folds
all_mod_preds = []
all_head_preds = []

print("🚀 Bắt đầu chạy Ensemble Inference 5 Folds trên tập Trial...")

for fold in range(5):
    model_path = f'{OUTPUT_DIR}/models/fold{fold}_best.pt'
    
    model = ModernBERTRegressor(
        CONFIG['model_name'],
        dropout=0.0  # Tắt Dropout hoàn toàn khi Inference
    ).to(device)
    
    # Load trọng số đã save của Fold
    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    model.eval()
    
    # FIX: Chỉ unpack 2 giá trị dự đoán từ evaluate
    res = evaluate(model, trial_loader, device)
    mod_pred, head_pred = res[0], res[1]
    
    all_mod_preds.append(mod_pred)
    all_head_preds.append(head_pred)
    
    print(f"  ✓ Đã hoàn thành Fold {fold}")
    
    del model
    torch.cuda.empty_cache()

# 3. Lấy trung bình cộng dự đoán (Averaging Ensemble)
trial_pred_mod = np.mean(all_mod_preds, axis=0)
trial_pred_head = np.mean(all_head_preds, axis=0)

# 4. Đánh giá chỉ số chính thức với Ground Truth
trial_rho_mod = spearmanr(df_trial['ModAvg'], trial_pred_mod).statistic
trial_rho_head = spearmanr(df_trial['HeadAvg'], trial_pred_head).statistic
trial_rmse_mod = np.sqrt(mean_squared_error(df_trial['ModAvg'], trial_pred_mod))
trial_rmse_head = np.sqrt(mean_squared_error(df_trial['HeadAvg'], trial_pred_head))

mean_trial_rho = (trial_rho_mod + trial_rho_head) / 2

print('\n=================== Trial Set Results (5-Fold Ensemble) ===================')
print(f'Mod  Spearman ρ: {trial_rho_mod:.4f}  |  RMSE: {trial_rmse_mod:.4f}')
print(f'Head Spearman ρ: {trial_rho_head:.4f}  |  RMSE: {trial_rmse_head:.4f}')
print(f'🔥 MEAN SPEARMAN ρ: {mean_trial_rho:.4f}')

In [ ]:
import zipfile

# Đường dẫn file
tsv_path = f'{OUTPUT_DIR}/submission/en-nn-trial-pred.tsv'
zip_path = f'{OUTPUT_DIR}/submission/submission.zip'

# Nén file TSV thành submission.zip (không chứa thư mục con)
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(tsv_path, arcname='en-nn-trial-pred.tsv')

print(f'\n📦 Đã đóng gói thành công file ZIP để nộp bài:')
print(f'👉 Đường dẫn: {zip_path}')

## Summary

**Model:** ModernBERT-base + regression head

**Training strategy (gradual unfreezing):**
- Phase 1 (epochs 1-2): Encoder FROZEN, head LR=1e-3 (learn good mapping from [CLS] features)
- Phase 2 (epochs 3-6): Encoder UNFROZEN, all LR=2e-5 (gentle adaptation, prevent forgetting)

**Ensemble:** Average predictions from 5 folds

### Comparison

| Model | OOF Mod rho | OOF Head rho | OOF Mean rho |
|-------|-------------|--------------|--------------|
| Word2Vec + Ridge | 0.1458 | 0.2468 | 0.1963 |
| ModernBERT + Ridge (frozen) | ? | ? | ? |
| ModernBERT fine-tuned (gradual unfreeze) | ? | ? | ? |